# Alarm clock

**Time:** 5 minutes

**What you build:** a drop zone. Drop any file on it (a PDF, a Word doc, a photo, a voice memo or a video) and it sends the file through a RocketRide pipeline that pulls out the text and shows it to you. Then you set it like an alarm clock: deployed to your organization's **Prod** team, awake from 9am to 6pm, Monday to Friday.

**Done when:** the **Set the alarm** cell prints your Prod deployment and its next start times, 9:00 AM on weekdays.

> Run the cells from top to bottom with **Shift+Enter**. Every cell is safe to run again.
> New to notebooks or haven't set up yet? Start with [SETUP.md](https://github.com/rocketride-org/rocketride-workshops/blob/main/SETUP.md).

## Preflight

Checks Python, the `rocketride` package, your API key and the connection to RocketRide Cloud. Fix anything marked ❌, then run it again.

In [ ]:
# Preflight: checks your machine. Prints one green "ready" line when everything passes.
import sys, urllib.error, urllib.request

problems, warnings = [], []

if sys.version_info < (3, 10):
    problems.append(f"Python {sys.version.split()[0]} is too old. Fix: install Python 3.10+ "
                    "and select it as this notebook's kernel (see SETUP.md).")

try:
    from rocketride import RocketRideClient
    if not RocketRideClient().get_apikey():
        warnings.append("No RocketRide API key in your environment or .env yet. "
                        "The Config cell below will ask for it.")
except ImportError:
    problems.append("The rocketride package is missing. Fix: run  %pip install rocketride  "
                    "in a new cell, then restart the kernel.")

try:
    urllib.request.urlopen("https://api.rocketride.ai", timeout=10)
except urllib.error.HTTPError:
    pass  # the server answered, so it's reachable
except Exception as e:
    problems.append(f"Can't reach api.rocketride.ai ({e}). Fix: check your internet "
                    "connection, VPN or proxy, then run this cell again.")

for w in warnings:
    print("⚠️ ", w)
if problems:
    for p in problems:
        print("❌", p)
    raise RuntimeError(f"Preflight failed: {len(problems)} problem(s). Fix them, then run this cell again.")
print("\033[32m✅ ready\033[0m")

## Config

Reads your RocketRide settings from the environment or a `.env` file next to this notebook. If your API key isn't set, it asks for it. Nothing is saved.

In [ ]:
# Config: the ONLY cell that reads environment variables. Every later cell uses CONFIG.
import getpass, os
from pathlib import Path

def read_dotenv(path=Path(".env")):
    values = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            key, sep, value = line.strip().partition("=")
            if sep and key.strip().startswith("ROCKETRIDE_"):
                values[key.strip()] = value.strip().strip("\"'")
    return values

env = read_dotenv()  # a .env file next to this notebook...
env.update({k: v for k, v in os.environ.items() if k.startswith("ROCKETRIDE_")})  # ...env wins
env.setdefault("ROCKETRIDE_URI", "https://api.rocketride.ai")
if not env.get("ROCKETRIDE_APIKEY"):
    previous = globals().get("CONFIG", {}).get("apikey")  # don't re-prompt on re-run
    env["ROCKETRIDE_APIKEY"] = previous or getpass.getpass("RocketRide API key: ").strip()

CONFIG = {
    "uri": env["ROCKETRIDE_URI"],
    "apikey": env["ROCKETRIDE_APIKEY"],
    "deploy_uri": env.get("ROCKETRIDE_DEPLOY_URI"),
    "deploy_apikey": env.get("ROCKETRIDE_DEPLOY_APIKEY"),
    "env": env,
}
print("Server: ", CONFIG["uri"])
print("API key:", "set" if CONFIG["apikey"] else "MISSING")
print("Deploy: ", CONFIG["deploy_uri"] or "not configured")

In [ ]:
# Connect: one shared RocketRide client for every cell below. Safe to re-run.
from rocketride import RocketRideClient

if "client" not in globals() or not client.is_connected():
    client = RocketRideClient(uri=CONFIG["uri"], auth=CONFIG["apikey"], env=CONFIG["env"])
    await client.connect()
print("Connected to", CONFIG["uri"])

async def start(pipe_path):
    """Start a pipeline, or attach to it if it's already running. Returns the task token."""
    result = await client.use(filepath=str(pipe_path), use_existing=True)
    return result["token"]

## Stage 1: Build the drop zone (1 min)

**Goal:** write the pipeline file. A **dropper** gives you a web page with a drop zone. **Parse** pulls the text out of documents. Two converters handle what parse can't read as text: **audio transcribe** turns speech in audio and video into text, and **OCR** reads the text in images. Every path ends in one **response** node that sends the text back.

**Watch for:** `Wrote build/alarm-clock.pipe` and a map of the five steps.

In [ ]:
# Stage 1: write the drop-zone pipeline to build/alarm-clock.pipe. Safe to re-run.
import json, uuid
from pathlib import Path

PIPE = Path("build/alarm-clock.pipe")
PIPE.parent.mkdir(exist_ok=True)

# Keep the same project id on re-runs, so a redeploy updates the same deployment.
old = json.loads(PIPE.read_text(encoding="utf-8")) if PIPE.exists() else {}
project_id = old.get("project_id") or str(uuid.uuid4())

def ui(x, y):
    return {"position": {"x": x, "y": y}, "measured": {"width": 150, "height": 66},
            "nodeType": "default", "formDataValid": True}

pipeline = {
    "components": [
        {   # The drop zone: a web page you drop files on
            "id": "dropper_1", "provider": "dropper",
            "config": {"hideForm": True, "mode": "Source", "parameters": {}, "type": "dropper"},
            "ui": ui(20, 200),
        },
        {   # Opens the file: text comes out as text, media comes out as audio, video or images
            "id": "parse_1", "provider": "parse", "config": {},
            "input": [{"lane": "tags", "from": "dropper_1"}],
            "ui": ui(240, 200),
        },
        {   # Speech in audio and video files becomes text
            "id": "audio_transcribe_1", "provider": "audio_transcribe",
            "config": {"profile": "default", "parameters": {}},
            "input": [{"lane": "audio", "from": "parse_1"}, {"lane": "video", "from": "parse_1"}],
            "ui": ui(460, 80),
        },
        {   # Text in images is read with OCR
            "id": "ocr_1", "provider": "ocr",
            "config": {"profile": "latin", "parameters": {}},
            "input": [{"lane": "image", "from": "parse_1"}],
            "ui": ui(460, 320),
        },
        {   # Sends all the text back to whoever dropped the file
            "id": "response_text_1", "provider": "response_text",
            "config": {"laneName": "text"},
            "input": [
                {"lane": "text", "from": "parse_1"},
                {"lane": "text", "from": "audio_transcribe_1"},
                {"lane": "text", "from": "ocr_1"},
            ],
            "ui": ui(680, 200),
        },
    ],
    "name": "Drop zone",
    "source": "dropper_1",
    "project_id": project_id,
    "viewport": {"x": 0, "y": 0, "zoom": 1},
    "version": 1,
}

PIPE.write_text(json.dumps(pipeline, indent=2), encoding="utf-8")
print("Wrote", PIPE)
for c in pipeline["components"]:
    inputs = ", ".join(f'{i["lane"]} from {i["from"]}' for i in c.get("input", []))
    print(f'  {c["id"]:<20} {"<- " + inputs if inputs else "(start)"}')

In [ ]:
# Skip ahead: run this only if the stage above didn't work
import json, uuid
from pathlib import Path

Path("build").mkdir(exist_ok=True)
ui = lambda x, y: {"position": {"x": x, "y": y}, "measured": {"width": 150, "height": 66},
                   "nodeType": "default", "formDataValid": True}
FINISHED = {
    "components": [
        {"id": "dropper_1", "provider": "dropper", "ui": ui(20, 200),
         "config": {"hideForm": True, "mode": "Source", "parameters": {}, "type": "dropper"}},
        {"id": "parse_1", "provider": "parse", "config": {}, "ui": ui(240, 200),
         "input": [{"lane": "tags", "from": "dropper_1"}]},
        {"id": "audio_transcribe_1", "provider": "audio_transcribe", "ui": ui(460, 80),
         "config": {"profile": "default", "parameters": {}},
         "input": [{"lane": "audio", "from": "parse_1"}, {"lane": "video", "from": "parse_1"}]},
        {"id": "ocr_1", "provider": "ocr", "ui": ui(460, 320),
         "config": {"profile": "latin", "parameters": {}},
         "input": [{"lane": "image", "from": "parse_1"}]},
        {"id": "response_text_1", "provider": "response_text", "ui": ui(680, 200),
         "config": {"laneName": "text"},
         "input": [{"lane": "text", "from": "parse_1"}, {"lane": "text", "from": "audio_transcribe_1"},
                   {"lane": "text", "from": "ocr_1"}]},
    ],
    "name": "Drop zone",
    "source": "dropper_1",
    "project_id": str(uuid.uuid4()),
    "viewport": {"x": 0, "y": 0, "zoom": 1},
    "version": 1,
}
Path("build/alarm-clock.pipe").write_text(json.dumps(FINISHED, indent=2), encoding="utf-8")
print("Wrote the finished pipeline to build/alarm-clock.pipe. Carry on with Stage 2.")

## Stage 2: Drop a file (2 min)

**Goal:** start the pipeline, then send it a file two ways. The cell sends a small sample file from the notebook and prints the text that comes back. It also prints a **Drop zone** link: open it in your browser and drop any file you have, such as a PDF, a screenshot or a voice memo.

**Watch for:** `Text back:` followed by the sample's text. To try your own file from the notebook, put its path in `MY_FILE` and run the cell again.

In [ ]:
# Stage 2: start the drop zone and send it a file. Safe to re-run.
from pathlib import Path

MY_FILE = ""  # optional: the path of any file, for example "~/Downloads/receipt.jpg"

info = await client.use(filepath="build/alarm-clock.pipe", use_existing=True)
token = info["token"]
DROP_ZONE = f'{CONFIG["uri"]}/dropper/{info["projectId"]}/{info["source"]}?auth={info["publicToken"]}'

sample = Path("build/sample.txt")
sample.write_text("Hello from the drop zone. If you can read this, your pipeline works.", encoding="utf-8")
file = Path(MY_FILE).expanduser() if MY_FILE else sample

[upload] = await client.send_files([str(file)], token)
if upload["action"] != "complete":
    raise RuntimeError(f'Sending {file.name} failed: {upload.get("error")}')
text = (upload.get("result") or {}).get("text") or []
print(f"Sent {file.name}")
print("Text back:", "\n".join(text) if isinstance(text, list) else text)
print()
print("Drop zone:", DROP_ZONE)

In [ ]:
# Skip ahead: run this only if the stage above didn't work
import json
from pathlib import Path

pipe = json.loads(Path("build/alarm-clock.pipe").read_text(encoding="utf-8"))
running = await client.get_task_token(pipe["project_id"], pipe["source"])
if running:
    await client.terminate(running)  # stop the stuck copy and start a fresh one

info = await client.use(filepath="build/alarm-clock.pipe")
token = info["token"]
DROP_ZONE = f'{CONFIG["uri"]}/dropper/{info["projectId"]}/{info["source"]}?auth={info["publicToken"]}'
print("Restarted the pipeline. Drop zone:", DROP_ZONE)

## Set the alarm (1 min)

Deploys your pipeline to your organization's **Prod** team and puts it on a schedule: it wakes at **9:00 AM** and runs for nine hours, until **6:00 PM**, **Monday to Friday**, in your local time. Outside those hours it's asleep.

Deploying needs an active RocketRide Cloud subscription and permission to deploy to the team. If your team has a different name, change `TEAM_NAME`. The times use your clock today, so run this cell again after the clocks change for daylight saving.

In [ ]:
# Set the alarm: deploy to the Prod team, awake 9am to 6pm, Monday to Friday. Safe to re-run.
import json
from datetime import datetime
from pathlib import Path

TEAM_NAME = "Prod"          # the team to deploy to
START, END = 9, 18          # 9:00 AM to 6:00 PM, your local time
WEEKDAYS = [1, 2, 3, 4, 5]  # cron days: 1 is Monday, 5 is Friday

pipeline = json.loads(Path("build/alarm-clock.pipe").read_text(encoding="utf-8"))

# Find the team by name.
org = (client.get_account_info() or {}).get("organization") or {}
teams = await client.account.list_teams(org["id"]) if org.get("id") else []
team = next((t for t in teams if t["name"].lower() == TEAM_NAME.lower()), None)
if not team:
    names = ", ".join(t["name"] for t in teams) or "none"
    raise RuntimeError(f'No team called "{TEAM_NAME}". Your teams: {names}. '
                       "Set TEAM_NAME to one of them, or ask an org admin to create it.")

# Publish a new version of the pipeline and point the team at it.
published = await client.deploy.publish(pipeline, comment="Alarm clock workshop", deploy_to=team["id"])

# The scheduler reads cron in UTC, so convert your local start time (and the days, if it crosses midnight).
monday = datetime(2024, 1, 1, START)  # 1 January 2024 was a Monday
utc_start = monday - datetime.now().astimezone().utcoffset()
shift = (utc_start.date() - monday.date()).days
days = ",".join(str(d) for d in sorted({(d + shift) % 7 for d in WEEKDAYS}))
cron = f"{utc_start.minute} {utc_start.hour} * * {days}"

dep = await client.deploy.set_schedule(pipeline["project_id"], pipeline["source"], cron, team["id"],
                                       ttl=(END - START) * 3600)
preview = await client.deploy.preview(cron, count=3)

print(f'Deployed "{pipeline["name"]}" v{published["artifact"]["version"]} to {team["name"]} ({dep["state"]})')
print(f"Alarm: {cron} (UTC), awake {END - START} hours each time")
for ts in preview.get("next", []):
    print("  wakes", datetime.fromtimestamp(ts).strftime("%a %d %b at %I:%M %p"))

# If it's awake right now, print the live drop zone.
live = await client.get_task_token(pipeline["project_id"], pipeline["source"], team_id=team["id"])
if live:
    notes = (await client.get_task_status(live)).get("notes") or []
    links = [n["button-link"] for n in notes if isinstance(n, dict) and n.get("button-link")]
    print("Live drop zone:", links[0].replace("{host}", CONFIG["uri"]) if links else "see Deployments in RocketRide")
else:
    print("Asleep right now. It wakes at the next time above.")

## What's next

- **Docs:** [docs.rocketride.org](https://docs.rocketride.org)
- **Help and community:** [RocketRide Discord](https://discord.gg/PMXrtenMsY)
- **Next module:** [Integrate RocketRide into an existing app](https://github.com/rocketride-org/rocketride-workshops/blob/main/modules/notepad-integration/notepad-integration.ipynb)